In [1]:
!pip install "numpy<2" -qqq
!pip install surprise -qqq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 24.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
p

In [ ]:
import pandas as pd;import polars as pl; import numpy as np; from surprise import Dataset, Reader, KNNWithMeans, accuracy
from surprise.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from surprise import KNNBasic, KNNWithMeans, KNNWithZScore, KNNBaseline
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import gdown, zipfile
url = 'https://drive.google.com/file/d/1HbSTBkFeczfTSkzm3KCIwzy6fOouTg_i/view?usp=sharing'
path = gdown.download(url, quiet=False, fuzzy=True)
with zipfile.ZipFile(path, 'r') as f: f.extractall('data')

#1: Подготовка данных

In [ ]:
r_l = pl.scan_csv('/content/data/ml-latest-small/ratings.csv').select(['userId', 'movieId', 'rating'])
m_l = pl.scan_csv('/content/data/ml-latest-small/movies.csv')
df = r_l.collect(); mov = m_l.collect()
r = Reader(rating_scale=(1, 5)); ds = Dataset.load_from_df(df.to_pandas(), r)
tr_set, tst_set = train_test_split(ds, test_size=0.2)

#2: Коллаборативная фильтрация (KNN)

In [ ]:
knn_variants = {
    'KNNBasic': KNNBasic(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNWithMeans': KNNWithMeans(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNWithZScore': KNNWithZScore(sim_options={'name': 'cosine', 'user_based': True}),
    'KNNBaseline': KNNBaseline(sim_options={'name': 'cosine', 'user_based': True})
}

knn_metrics = []

for name, algo in knn_variants.items():
    algo.fit(tr_set)
    predictions = algo.test(tst_set)

    y_t_knn = [p.r_ui for p in predictions]
    y_p_knn = [p.est for p in predictions]

    metrics = [
        r2_score(y_t_knn, y_p_knn),
        np.sqrt(mean_squared_error(y_t_knn, y_p_knn)),
        mean_absolute_error(y_t_knn, y_p_knn),
        mean_squared_error(y_t_knn, y_p_knn)
    ]
    knn_metrics.append(metrics)

res_knn = pd.DataFrame(knn_metrics, columns=['R2', 'RMSE', 'MAE', 'MSE'], index=knn_variants.keys())
display(res_knn.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
                     .background_gradient(cmap='RdYlGn', subset=['R2']))

best_knn_name = res_knn['R2'].idxmax()
print(f"Лучшая модель для гибрида: {best_knn_name}")

best_algo = knn_variants[best_knn_name]
best_pred = best_algo.test(tst_set)
y_p = [p.est for p in best_pred]
met_cf = knn_metrics[list(knn_variants.keys()).index(best_knn_name)]

y_t = [p[2] for p in tst_set] # p[2] — это оценка (rating) в кортеже (uid, iid, rating)



Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.


,R2,RMSE,MAE,MSE
KNNBasic,0.138023,0.969451,0.746683,0.939834
KNNWithMeans,0.255758,0.900813,0.689559,0.811465
KNNWithZScore,0.262255,0.896873,0.682274,0.804381
KNNBaseline,0.294693,0.876934,0.671601,0.769013


Лучшая модель для гибрида: KNNBaseline


#3: Контентная фильтрация (жанры)

In [ ]:
#3. Реализация контентной фильтрации
g_e = mov.with_columns(pl.col("genres").str.split("|")).explode("genres") \
         .pivot(index="movieId", on="genres", values="genres", aggregate_function="len").fill_null(0)
#Приводим к Int8 только столбцы жанров (кроме movieId)
cols = [c for c in g_e.columns if c != 'movieId']
g_e = g_e.with_columns([pl.col(c).cast(pl.Int8) for c in cols])

# 1. Считаем среднее по каждому пользователю
user_means = df.group_by("userId").agg(pl.col("rating").mean().alias("avg_rating"))
df_norm = df.join(user_means, on="userId").with_columns(
    (pl.col("rating") - pl.col("avg_rating")).alias("norm_rating")
)

# 2. Создаем профили пользователей на основе нормализованных оценок
u_p = df_norm.join(g_e, on="movieId").group_by("userId").agg(
    # Взвешиваем жанры фильма на отклонение оценки от среднего
    [(pl.col(c) * pl.col("norm_rating")).sum() / (pl.col("norm_rating").std() + 1e-9)
     for c in cols]
)

u_d = dict(zip(u_p["userId"].to_list(), u_p.drop("userId").to_numpy()))
m_d = dict(zip(g_e["movieId"].to_list(), g_e.drop("movieId").to_numpy()))

# 3. Предикт (теперь результат будет колебаться вокруг 0)
def cb_p(u, m):
    try:
        # Считаем схожесть векторов
        u_vec = u_d[u]
        m_vec = m_d[m]
        # Скалярное произведение нормализованных векторов
        score = np.dot(u_vec, m_vec) / (np.linalg.norm(u_vec) * np.linalg.norm(m_vec) + 1e-9)
        # Возвращаем к шкале 1-5, прибавляя среднее (например, 3.5)
        return 3.5 + score * 1.5
    except:
        return 3.5


pred_cb = [cb_p(p[0], p[1]) for p in tst_set];
y_p_cb = np.clip(pred_cb, 1, 5)
met_cb = [r2_score(y_t, y_p_cb), np.sqrt(mean_squared_error(y_t, y_p_cb)), mean_absolute_error(y_t, y_p_cb), mean_squared_error(y_t, y_p_cb)]
display(met_cb)

[0.023395784176757473,
 1.0318986407212298,
 0.8147844001241185,
 1.0648148047223216]

#4: Гибридизация и оценка

In [ ]:
# Теперь запускаем перебор alpha (ваш код ниже)
y_p_knn_arr = np.array(y_p)
y_p_cb_arr = np.array(y_p_cb)
y_t_arr = np.array(y_t)

alphas = np.linspace(0, 1, 21)
grid_data = []

for a in alphas:
    y_p_h_tmp = a * np.array(y_p) + (1 - a) * y_p_cb
    y_p_h_tmp = np.clip(y_p_h_tmp, 1, 5)
    grid_data.append([
        a,
        r2_score(y_t, y_p_h_tmp),
        np.sqrt(mean_squared_error(y_t, y_p_h_tmp)),
        mean_absolute_error(y_t, y_p_h_tmp),
        mean_squared_error(y_t, y_p_h_tmp)
    ])

# Вывод таблицы перебора для анализа
df_grid = pd.DataFrame(grid_data, columns=['alpha', 'R2', 'RMSE', 'MAE', 'MSE'])
display(df_grid.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
               .background_gradient(cmap='RdYlGn', subset=['R2'])
               .format({'alpha': '{:.2f}'}))

# 2. Фиксация лучшего 'a' на основе максимального R2
best_idx = df_grid['R2'].idxmax()
a = df_grid.loc[best_idx, 'alpha']

# 3. Финальный расчет с зафиксированным 'a'
y_p_h = a * np.array(y_p) + (1 - a) * y_p_cb
y_p_h = np.clip(y_p_h, 1, 5)

met_h = [
    r2_score(y_t, y_p_h),
    np.sqrt(mean_squared_error(y_t, y_p_h)),
    mean_absolute_error(y_t, y_p_h),
    mean_squared_error(y_t, y_p_h)
]

# 4. Итоговое сравнение моделей
res = pd.DataFrame([met_cf, met_cb, met_h],
                   columns=['R2', 'RMSE', 'MAE', 'MSE'],
                   index=['KNN', 'Content', 'Hybrid'])

print(f"\nЛучшее значение alpha зафиксировано: {a}")
display(res.style.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE', 'MSE'])
                 .background_gradient(cmap='RdYlGn', subset=['R2']))


,alpha,R2,RMSE,MAE,MSE
0,0.00,0.023396,1.031899,0.814784,1.064815
1,0.05,0.064011,1.010213,0.796615,1.020531
2,0.10,0.101779,0.989622,0.779298,0.979352
3,0.15,0.136699,0.970195,0.762818,0.941278
4,0.20,0.168772,0.952002,0.747310,0.906308
5,0.25,0.197997,0.935116,0.732853,0.874443
6,0.30,0.224375,0.919610,0.719523,0.845682
7,0.35,0.247906,0.905553,0.707245,0.820026
8,0.40,0.268589,0.893014,0.696068,0.797474
9,0.45,0.286425,0.882059,0.686025,0.778027



Лучшее значение alpha зафиксировано: 0.75


,R2,RMSE,MAE,MSE
KNN,0.294693,0.876934,0.671601,0.769013
Content,0.023396,1.031899,0.814784,1.064815
Hybrid,0.333646,0.852374,0.655907,0.726542


#ИНДЗ

In [1]:
import asyncio; from curl_cffi import requests; from bs4 import BeautifulSoup; import polars as pl; import os; import gc; import json; import shutil; import subprocess
import numpy as np; import pandas as pd; from surprise import Dataset, Reader, KNNBaseline; from surprise.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error; from sklearn.feature_extraction.text import CountVectorizer
import random; import re
k_k=os.environ.get('KAGGLE_KEY', 'xxx'); k_u=os.environ.get('KAGGLE_USERNAME', 'xxx')
d_i=f"{k_u}/aoty-metalcore-dataset"; d_p='/content/kaggle_aoty'; os.makedirs(d_p, exist_ok=True)


In [9]:
import asyncio; from curl_cffi import requests; from bs4 import BeautifulSoup; import polars as pl; import os; import gc; import re; import random
import datetime
s_m=asyncio.Semaphore(2); a_c={}

async def fetch_html_from_url(url, r_c=3):
    async with s_m:
        for i in range(r_c):
            await asyncio.sleep(random.uniform(2.0, 4.0))
            try:
                async with requests.AsyncSession(impersonate="chrome120") as s:
                    r=await s.get(url, timeout=15, allow_redirects=True)
                    print(f"HTTP {r.status_code} | GET {url}")
                    if r.status_code==200 and "Just a moment" not in r.text: return 200, r.text, r.url
                    if r.status_code in (403, 429):
                        sl=5*(i+1)
                        print(f"WAF БЛОКИРОВКА. ШТРАФНОЕ ОЖИДАНИЕ {sl}с")
                        await asyncio.sleep(sl)
            except Exception as e:
                print(f"HTTP ERR | {e}"); await asyncio.sleep(2)
        return 0, "", url

async def extract_seed_users_from_album(album_slug):
    u_s=set(); print(f"=== СБОР СИДОВ: {album_slug} ===")
    for p in range(1, 16):
        u=f"https://www.albumoftheyear.org/album/{album_slug}/user-reviews/?p={p}" if p>1 else f"https://www.albumoftheyear.org/album/{album_slug}/user-reviews/"
        c, h, _=await fetch_html_from_url(u)
        if c!=200: break
        u_f=re.findall(r'href="/user/([^/]+)/"', h)
        if not u_f: break
        for x in u_f:
            if x not in ('thumbs', 'login', 'reviews', 'ratings'): u_s.add(x)
        print(f"СИДОВ ИЗВЛЕЧЕНО: {len(u_s)} (СТРАНИЦА {p})")
    return list(u_s)

async def extract_album_ratings_for_user(user_id):
    r_l=[]; print(f"--- ПАРСИНГ ПРОФИЛЯ: {user_id} ---")
    for p in range(1, 6):
        u=f"https://www.albumoftheyear.org/user/{user_id}/reviews/?p={p}" if p>1 else f"https://www.albumoftheyear.org/user/{user_id}/reviews/"
        c, h, _=await fetch_html_from_url(u)
        if c!=200: break
        sp=BeautifulSoup(h, 'lxml'); r_d=sp.find_all('div', class_=re.compile(r'^rating$'))
        if not r_d: break
        for d in r_d:
            try: r_v=int(d.text.strip())
            except ValueError: continue
            pr=d.parent; a_id=None
            for _ in range(6):
                if not pr or pr.name=='body': break
                a_t=pr.find('a', href=re.compile(r'/album/\d+-'))
                if a_t:
                    m=re.search(r'/album/(\d+)-', a_t['href'])
                    if m: a_id=int(m.group(1)); break
                pr=pr.parent
            if a_id and not any(x['a_id']==a_id for x in r_l): r_l.append({'u_id': user_id, 'a_id': a_id, 'rating': r_v})
    print(f"ПРОФИЛЬ {user_id} ЗАВЕРШЕН. ОЦЕНОК: {len(r_l)}")
    return r_l

async def extract_metadata_for_album(album_id):
    if album_id in a_c: return a_c[album_id]
    u=f"https://www.albumoftheyear.org/album/{album_id}"
    c, h, _=await fetch_html_from_url(u)
    if c!=200: return None
    sp=BeautifulSoup(h, 'lxml'); t_e=sp.find('h1'); g_e=sp.find_all('a', href=lambda x: x and '/genre/' in x)
    if t_e:
        d={'a_id': album_id, 'title': t_e.text.strip(), 'genres': '|'.join([g.text for g in g_e]) if g_e else 'Unknown'}
        a_c[album_id]=d; return d
    return None

async def execute_full_pipeline(album_slug):
    u_l=await extract_seed_users_from_album(album_slug)
    if not u_l: return
    a_r=[]; c_z=3
    for i in range(0, len(u_l), c_z):
        b=u_l[i:i+c_z]; t=[extract_album_ratings_for_user(u) for u in b]
        r=await asyncio.gather(*t)
        for i_r in r: a_r.extend(i_r)
    if not a_r: print("МАТРИЦА ПУСТА."); return
    r_df=pl.DataFrame(a_r); r_df.write_parquet("u_r.parquet")
    u_a=r_df['a_id'].unique().to_list(); a_m=[]; c_z_a=5
    print(f"=== СБОР МЕТАДАННЫХ ДЛЯ {len(u_a)} АЛЬБОМОВ ===")
    for i in range(0, min(len(u_a), 1500), c_z_a):
        b=u_a[i:i+c_z_a]; t=[extract_metadata_for_album(a) for a in b]
        r=await asyncio.gather(*t)
        for i_m in r:
            if i_m: a_m.append(i_m)
    if a_m: pl.DataFrame(a_m).write_parquet("a_m.parquet")
    print("=== КОНВЕЙЕР ЗАВЕРШЕН ===")
    del a_r, a_m, r_df; gc.collect()



def upload_dataset_to_kaggle():
    if not os.path.exists("users_ratings.parquet"): return
    with open(os.path.join(d_p, 'dataset-metadata.json'), 'w') as f: json.dump({"title": "AOTY_Metalcore", "id": d_i, "licenses": [{"name": "CC0-1.0"}]}, f)
    for f in ["users_ratings.parquet", "albums_metadata.parquet"]:
        if os.path.exists(f): shutil.copy(f, os.path.join(d_p, f))
    c=subprocess.run(["kaggle", "datasets", "status", d_i], capture_output=True, text=True)
    if c.returncode==0: subprocess.run(["kaggle", "datasets", "version", "-p", d_p, "-m", f"Update: {datetime.datetime.now()}"])
    else: subprocess.run(["kaggle", "datasets", "create", "-p", d_p])

def train_hybrid_recsys():
    if not os.path.exists("users_ratings.parquet") or not os.path.exists("albums_metadata.parquet"):
        print("Отсутствуют .parquet файлы. Запуск остановлен.")
        return

    df=pl.read_parquet("users_ratings.parquet"); mov=pl.read_parquet("albums_metadata.parquet")
    p_df=df.to_pandas()
    r=Reader(rating_scale=(0, 100)); ds=Dataset.load_from_df(p_df[['u_id', 'a_id', 'rating']], r)
    tr_set, tst_set=train_test_split(ds, test_size=0.2)

    k_m=KNNBaseline(sim_options={'name': 'pearson_baseline', 'user_based': True}); k_m.fit(tr_set)
    p_k=k_m.test(tst_set)
    y_t=[p.r_ui for p in p_k]; y_p_k=np.clip([p.est for p in p_k], 0, 100)

    g_e=mov.with_columns(pl.col("genres").str.split("|")).explode("genres").pivot(index="a_id", on="genres", values="genres", aggregate_function="len").fill_null(0)
    c_g=[c for c in g_e.columns if c!='a_id']
    g_e=g_e.with_columns([pl.col(c).cast(pl.Int8) for c in c_g])

    u_m=df.group_by("u_id").agg(pl.col("rating").mean().alias("a_r"))
    d_n=df.join(u_m, on="u_id").with_columns((pl.col("rating")-pl.col("a_r")).alias("n_r"))
    u_p=d_n.join(g_e, on="a_id").group_by("u_id").agg([(pl.col(c)*pl.col("n_r")).sum()/(pl.col("n_r").std()+1e-9) for c in c_g])

    u_d=dict(zip(u_p["u_id"].to_list(), u_p.drop("u_id").to_numpy())); m_d=dict(zip(g_e["a_id"].to_list(), g_e.drop("a_id").to_numpy()))

    def cb_p(u, m):
        try:
            u_v=u_d[u]; m_v=m_d[m]
            s=np.dot(u_v, m_v)/(np.linalg.norm(u_v)*np.linalg.norm(m_v)+1e-9)
            return 50.0 + s*25.0
        except: return 50.0

    y_p_cb=np.clip([cb_p(p[0], p[1]) for p in tst_set], 0, 100)

    b_a=0; b_r2=-np.inf
    for a in np.linspace(0, 1, 21):
        yh=np.clip(a*np.array(y_p_k)+(1-a)*np.array(y_p_cb), 0, 100)
        cr2=r2_score(y_t, yh)
        if cr2>b_r2: b_r2=cr2; b_a=a

    y_h_f=np.clip(b_a*np.array(y_p_k)+(1-b_a)*np.array(y_p_cb), 0, 100)

    def gm_f(y, p): return [r2_score(y, p), np.sqrt(mean_squared_error(y, p)), mean_absolute_error(y, p), mean_squared_error(y, p)]
    res=pd.DataFrame([gm_f(y_t, y_p_k), gm_f(y_t, y_p_cb), gm_f(y_t, y_h_f)], columns=['R2', 'RMSE', 'MAE', 'MSE'], index=['KNN', 'Content', 'Hybrid'])
    print(f"\nОптимальная альфа: {b_a:.2f}")
    print(res)

In [10]:
await execute_full_pipeline("290576-bring-me-the-horizon-post-human-survival-horror")

=== СБОР СИДОВ: 290576-bring-me-the-horizon-post-human-survival-horror ===
HTTP 403 | GET https://www.albumoftheyear.org/album/290576-bring-me-the-horizon-post-human-survival-horror/user-reviews/
WAF БЛОКИРОВКА. ШТРАФНОЕ ОЖИДАНИЕ 5с
HTTP 200 | GET https://www.albumoftheyear.org/album/290576-bring-me-the-horizon-post-human-survival-horror/user-reviews/
СИДОВ ИЗВЛЕЧЕНО: 25 (СТРАНИЦА 1)
HTTP 403 | GET https://www.albumoftheyear.org/album/290576-bring-me-the-horizon-post-human-survival-horror/user-reviews/?p=2
WAF БЛОКИРОВКА. ШТРАФНОЕ ОЖИДАНИЕ 5с
HTTP 403 | GET https://www.albumoftheyear.org/album/290576-bring-me-the-horizon-post-human-survival-horror/user-reviews/?p=2
WAF БЛОКИРОВКА. ШТРАФНОЕ ОЖИДАНИЕ 10с
HTTP 403 | GET https://www.albumoftheyear.org/album/290576-bring-me-the-horizon-post-human-survival-horror/user-reviews/?p=2
WAF БЛОКИРОВКА. ШТРАФНОЕ ОЖИДАНИЕ 15с
--- ПАРСИНГ ПРОФИЛЯ: lenyah ---
--- ПАРСИНГ ПРОФИЛЯ: desaguadouro ---
--- ПАРСИНГ ПРОФИЛЯ: andiov1 ---
HTTP 200 | GET https:/

CancelledError: 

In [ ]:
upload_dataset_to_kaggle()

In [ ]:
train_hybrid_recsys()

Отсутствуют .parquet файлы. Запуск остановлен.
